In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from sklearn.feature_selection import f_classif
from scipy.stats import kruskal

sys.path.append(str(Path("..").resolve()))
from fleetsense.config import DATA_DATASET

In [ ]:
pddf = pd.read_csv(DATA_DATASET / "vessel_weekly_features.csv")

pddf.value_counts("ship_type")

In [ ]:
print(pddf.columns)

In [ ]:
# Feature groups — each will become one figure with a grid of boxplots
selected_features = [
    "mean_moving_speed",
    "sog_p10",
    "sog_median",
    "sog_p90",
    "frac_time_slow",
    "frac_time_fast",
    "rot_mean_abs",
    "rot_std",
    "heading_cog_diff_mean",
    "fishing_ratio",
    "anchor_ratio",
    "underway_engine_ratio",
    "moored_ratio",
    "length",
    "width",
    "max_draught",
    "draught_variability",
    "length_beam_ratio",
    "draught_length_ratio",
    "bbox_area",
    "n_pings",
    "mean_ping_interval_seconds",
]

feature_groups = {
    "Speed Profile": [
        "mean_moving_speed",
        "max_speed",
        "std_speed",
        "sog_p10",
        "sog_median",
        "sog_p90",
        "frac_time_slow",
        "frac_time_fast",
        "sog_bimodality",
    ],
    "Course & Heading": [
        "cog_variability",
        "rot_mean_abs",
        "rot_std",
        "heading_cog_diff_mean",
    ],
    "Navigational Status": [
        "fishing_ratio",
        "anchor_ratio",
        "underway_engine_ratio",
        "moored_ratio",
        "n_nav_statuses",
    ],
    "Physical Dimensions": [
        "length",
        "width",
        "max_draught",
        "min_draught",
        "draught_variability",
        "length_beam_ratio",
        "draught_length_ratio",
    ],
    "Spatial & Trajectory": [
        "lat_std",
        "lon_std",
        "bbox_area",
        "straightness_index",
    ],
    "Temporal & Ping": [
        "n_pings",
        "time_span_seconds",
        "mean_ping_interval_seconds",
    ],
}

In [ ]:
pddf = pddf.dropna()
X = pddf[selected_features]
y = pddf["ship_type"]
# ANOVA F-stat (sklearn gives you this directly, vectorized across all features)
f_scores, p_values = f_classif(X, y)

# Kruskal-Wallis (per feature, manual loop since scipy doesn't vectorize across columns)
results = {}
for col in X.columns:
    groups = [X[col][y == cls] for cls in y.unique()]
    h_stat, p_val = kruskal(*groups)
    results[col] = (h_stat, p_val)

results_f = pd.DataFrame([selected_features, f_scores, p_values])
results

In [ ]:
# pddf= pddf[pddf["ship_type"].isin([ "Fishing", "Tug", "Passenger"])]  # Filter for specific ship types
class_column = "ship_type"  # Replace with the actual column name for vessel class
palette = {
    "Cargo": "#1C08F300",  # purple
    "Tanker": "#055519",  # teal
    "Fishing": "#F1460C",  # coral
    "Tug": "#E015B4",  # blue
    "Passenger": "#0FEE1A",  # pink
}


def plot_feature_group(df, group_name: str, features: list[str]) -> None:
    # Only keep features that actually exist in the dataframe
    features = [f for f in features if f in df.columns]
    if not features:
        print(f"Skipping '{group_name}' — no matching columns found.")
        return

    n = len(features)
    ncols = 3
    nrows = (n + ncols - 1) // ncols  # ceiling division

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
    axes = axes.flatten()

    for i, feat in enumerate(features):
        ax = axes[i]
        sns.boxplot(
            data=df,
            x=class_column,
            hue=class_column,
            y=feat,
            palette=palette,
            order=list(palette.keys()),
            showfliers=False,  # hide outliers for readability; set True to show them
            linewidth=0.8,
            ax=ax,
        )
        ax.set_title(feat, fontsize=11)
        ax.set_xlabel("")
        ax.set_ylabel("")

        ax.tick_params(axis="x", rotation=25)

    # Hide any unused subplot slots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f"Feature distributions by ship type — {group_name}", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
pddf.groupby("ship_type")["bbox_area"].describe()

In [ ]:
plot_feature_group(pddf, "All features", selected_features)

In [ ]:
matrix = pddf[selected_features].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5,
)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# 'common_norm=False' normalizes each class independently so you can see relative shapes
sns.displot(
    data=pddf,
    x="draught_length_ratio",
    hue="ship_type",
    kind="kde",
    fill=True,
    common_norm=False,
)
plt.show()

In [ ]:
# 'common_norm=False' normalizes each class independently so you can see relative shapes
sns.displot(data=pddf, x="n_pings", hue="ship_type", kind="kde", fill=True, common_norm=False)
plt.show()